In [1]:
import os
import time
from datetime import datetime
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F

In [2]:
LAKE = os.environ.get("LAKE_PATH", "file:///data-lake/data-lake")
BRONZE = LAKE + "/bronze"
SILVER = LAKE + "/silver"
REJECTED = SILVER + "/_rejected"
MONITORING = LAKE + "/monitoring/silver_quality"
RUN_ID = datetime.now().strftime("%Y%m%d%H%M%S")
JOB = "silver_job"

spark = (
    SparkSession.builder
    .appName("silver")
    .master("local[*]")
    .config("spark.driver.memory", "5g")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)
c = F.col

In [3]:
SPECS = {
    "FactOnlineSales": [
        ("OnlineSalesKey", "int"), ("DateKey", "timestamp"), ("StoreKey", "int"), ("ProductKey", "int"),
        ("PromotionKey", "int"), ("CurrencyKey", "int"), ("CustomerKey", "int"),
        ("SalesOrderNumber", "string"), ("SalesOrderLineNumber", "int"),
        ("SalesQuantity", "int"), ("ReturnQuantity", "int"), ("DiscountQuantity", "int"),
        ("UnitCost", "decimal(19,4)"), ("UnitPrice", "decimal(19,4)"), ("SalesAmount", "decimal(19,4)"),
        ("ReturnAmount", "decimal(19,4)"), ("DiscountAmount", "decimal(19,4)"), ("TotalCost", "decimal(19,4)"),
    ],
    "DimCustomer": [
        ("CustomerKey", "int"), ("GeographyKey", "int"), ("FirstName", "string"), ("MiddleName", "string"),
        ("LastName", "string"), ("BirthDate", "timestamp"), ("MaritalStatus", "string"), ("Gender", "string"),
        ("YearlyIncome", "decimal(19,4)"), ("TotalChildren", "int"), ("Education", "string"),
        ("Occupation", "string"), ("DateFirstPurchase", "timestamp"), ("CustomerType", "string"),
        ("CompanyName", "string"),
    ],
    "DimProduct": [
        ("ProductKey", "int"), ("ProductName", "string"), ("ProductDescription", "string"),
        ("ProductSubcategoryKey", "int"), ("Manufacturer", "string"), ("BrandName", "string"),
        ("ClassName", "string"), ("ColorName", "string"), ("Size", "string"), ("Weight", "decimal(18,4)"),
        ("UnitCost", "decimal(19,4)"), ("UnitPrice", "decimal(19,4)"), ("StockTypeName", "string"),
        ("Status", "string"), ("AvailableForSaleDate", "timestamp"), ("StopSaleDate", "timestamp"),
    ],
    "DimStore": [
        ("StoreKey", "int"), ("GeographyKey", "int"), ("StoreType", "string"), ("StoreName", "string"),
        ("Status", "string"), ("OpenDate", "timestamp"), ("CloseDate", "timestamp"),
        ("EmployeeCount", "int"), ("SellingAreaSize", "decimal(18,2)"), ("ZipCode", "string"),
    ],
}
LINEAGE = ["_ingestion_ts", "_source_system", "_source_table"]      # added in Bronze, kept all the way


def enforce_schema(df, spec):
    have = set(x.lower() for x in df.columns)
    missing = [n for n, _ in spec if n.lower() not in have]
    if missing:
        raise ValueError("Bronze is missing columns: %s" % missing)
    cols = [c(n).cast(t).alias(n) for n, t in spec]
    cols += [c(x) for x in LINEAGE if x in df.columns]
    return df.select(*cols)


def clean_strings(df, spec):
    """trim every text column, and turn blank text into NULL"""
    for n, t in spec:
        if t == "string":
            trimmed = F.trim(c(n))
            df = df.withColumn(n, F.when(trimmed == "", F.lit(None).cast("string")).otherwise(trimmed))
    return df


def mark_dups(df, keys, flag):
    """add a boolean column that is true for every row whose key appears more than once"""
    dups = (df.groupBy(*keys).count().filter(c("count") > 1)
            .select(*(list(keys) + [F.lit(True).alias(flag)])))
    return df.join(F.broadcast(dups), list(keys), "left")


def negative(cols):
    return [(c(n) < 0, n + " is negative", n) for n in cols]


def first_failing(rules, idx):
    """idx 1 = the reason text, idx 2 = the column. The first rule that fails wins (same as the SSIS CASE)."""
    expr = None
    for rule in rules:
        piece = F.lit(rule[idx])
        expr = F.when(rule[0], piece) if expr is None else expr.when(rule[0], piece)
    return expr


In [4]:
def prep_fact_online_sales(df):
    df = (df.withColumn("SaleDate", F.to_date("DateKey"))
            .withColumn("DateKey", F.date_format("SaleDate", "yyyyMMdd").cast("int"))
            .withColumn("SaleYear", F.year("SaleDate"))
            .withColumn("SaleMonth", F.month("SaleDate")))
    df = mark_dups(df, ["OnlineSalesKey"], "_dup_key")
    return mark_dups(df, ["SalesOrderNumber", "SalesOrderLineNumber"], "_dup_line")


def prep_dim_customer(df):
    g, m = F.upper(c("Gender")), F.upper(c("MaritalStatus"))
    df = (df.withColumn("Gender", F.when(g.isin("M", "MALE"), "Male").when(g.isin("F", "FEMALE"), "Female").otherwise("Unknown"))
            .withColumn("MaritalStatus", F.when(m.isin("M", "MARRIED"), "Married").when(m.isin("S", "SINGLE"), "Single").otherwise("Unknown"))
            .withColumn("BirthDate", F.to_date("BirthDate"))
            .withColumn("DateFirstPurchase", F.to_date("DateFirstPurchase")))
    full = F.trim(F.concat_ws(" ", c("FirstName"), c("MiddleName"), c("LastName")))      # concat_ws skips NULLs
    df = df.withColumn("FullName", F.when(full == "", c("CompanyName")).otherwise(full))
    df = df.fillna({"Education": "Unknown", "Occupation": "Unknown", "CompanyName": "Unknown"})
    return mark_dups(df, ["CustomerKey"], "_dup_key")


def prep_dim_product(df):
    df = (df.withColumn("AvailableForSaleDate", F.to_date("AvailableForSaleDate"))
            .withColumn("StopSaleDate", F.to_date("StopSaleDate")))                    # NULL = still on sale
    df = df.fillna({"Manufacturer": "Unknown", "BrandName": "Unknown", "ClassName": "Unknown",
                    "ColorName": "Unknown", "StockTypeName": "Unknown"})
    return mark_dups(df, ["ProductKey"], "_dup_key")


def prep_dim_store(df):
    df = (df.withColumn("OpenDate", F.to_date("OpenDate")).withColumn("CloseDate", F.to_date("CloseDate")))
    df = df.fillna({"ZipCode": "Unknown"})
    return mark_dups(df, ["StoreKey"], "_dup_key")


In [5]:
FACT_RULES = (
    [(c("OnlineSalesKey").isNull(), "OnlineSalesKey is NULL", "OnlineSalesKey"),
     (c("SaleDate").isNull(), "DateKey is NULL", "DateKey"),
     (c("StoreKey").isNull(), "StoreKey is NULL", "StoreKey"),
     (c("ProductKey").isNull(), "ProductKey is NULL", "ProductKey"),
     (c("CustomerKey").isNull(), "CustomerKey is NULL", "CustomerKey"),
     (c("SalesOrderNumber").isNull(), "SalesOrderNumber is NULL", "SalesOrderNumber"),
     (c("SalesOrderLineNumber").isNull(), "SalesOrderLineNumber is NULL", "SalesOrderLineNumber")]
    + negative(["SalesQuantity", "ReturnQuantity", "DiscountQuantity"])
    + [(c("ReturnQuantity") > c("SalesQuantity"), "ReturnQuantity exceeds SalesQuantity", "ReturnQuantity"),
       (c("DiscountQuantity") > c("SalesQuantity"), "DiscountQuantity exceeds SalesQuantity", "DiscountQuantity")]
    + negative(["SalesAmount", "ReturnAmount", "DiscountAmount", "TotalCost", "UnitCost", "UnitPrice"])
    + [(c("DiscountAmount") > c("SalesQuantity") * c("UnitPrice"), "DiscountAmount exceeds the gross amount", "DiscountAmount"),
       (c("SaleDate") > F.current_date(), "DateKey is in the future", "DateKey"),
       (c("_dup_key").isNotNull(), "Duplicate OnlineSalesKey", "OnlineSalesKey"),
       (c("_dup_line").isNotNull(), "Duplicate order line", "SalesOrderNumber, SalesOrderLineNumber")]
)
CUSTOMER_RULES = [
    (c("CustomerKey").isNull(), "CustomerKey is NULL", "CustomerKey"),
    (c("FullName").isNull(), "FullName is empty", "FullName"),
    (c("BirthDate") > F.current_date(), "BirthDate is in the future", "BirthDate"),
    (c("DateFirstPurchase") > F.current_date(), "DateFirstPurchase is in the future", "DateFirstPurchase"),
] + negative(["YearlyIncome", "TotalChildren"]) + [
    (c("_dup_key").isNotNull(), "Duplicate CustomerKey", "CustomerKey"),
]
PRODUCT_RULES = [
    (c("ProductKey").isNull(), "ProductKey is NULL", "ProductKey"),
    (c("ProductName").isNull(), "ProductName is empty", "ProductName"),
] + negative(["UnitPrice", "UnitCost", "Weight"]) + [
    (c("_dup_key").isNotNull(), "Duplicate ProductKey", "ProductKey"),
]
STORE_RULES = [
    (c("StoreKey").isNull(), "StoreKey is NULL", "StoreKey"),
    (c("StoreName").isNull(), "StoreName is empty", "StoreName"),
] + negative(["EmployeeCount", "SellingAreaSize"]) + [
    (c("_dup_key").isNotNull(), "Duplicate StoreKey", "StoreKey"),
]

TABLES = {
    "FactOnlineSales": dict(key="OnlineSalesKey", prep=prep_fact_online_sales, rules=FACT_RULES,
                            partition=["SaleYear", "SaleMonth"]),
    "DimCustomer": dict(key="CustomerKey", prep=prep_dim_customer, rules=CUSTOMER_RULES, partition=None),
    "DimProduct": dict(key="ProductKey", prep=prep_dim_product, rules=PRODUCT_RULES, partition=None),
    "DimStore": dict(key="StoreKey", prep=prep_dim_store, rules=STORE_RULES, partition=None),
}


In [6]:
def process(table):
    t0 = time.time()
    cfg, spec = TABLES[table], SPECS[table]

    df = clean_strings(enforce_schema(spark.read.parquet(BRONZE + "/" + table), spec), spec)
    df = cfg["prep"](df)
    flagged = (df.withColumn("_reason", first_failing(cfg["rules"], 1))
                 .withColumn("_error_col", first_failing(cfg["rules"], 2))
                 .persist(StorageLevel.MEMORY_AND_DISK))

    total = flagged.count()
    rejected_n = flagged.filter(c("_reason").isNotNull()).count()
    valid_n = total - rejected_n

    helper = [x for x in flagged.columns if x.startswith("_dup") or x in ("_reason", "_error_col")]
    valid = (flagged.filter(c("_reason").isNull()).drop(*helper)
             .withColumn("_silver_ts", F.current_timestamp()))
    first = [n for n, _ in spec]                                  # keep the schema order (the joins move columns around)
    valid = valid.select(*(first + [x for x in valid.columns if x not in first]))
    out = SILVER + "/" + table
    if cfg["partition"]:
        (valid
            .repartition(16)
            .write
            .mode("overwrite")
            .option("compression", "snappy")
            .partitionBy(*cfg["partition"])
            .parquet(out))
    else:
        (valid
            .coalesce(1)
            .write
            .mode("overwrite")
            .option("compression", "snappy")
            .parquet(out))

    rejected = flagged.filter(c("_reason").isNotNull()).select(
        F.coalesce(c(cfg["key"]).cast("string"), F.lit("(null key)")).alias("RecordID"),
        F.lit(table).alias("SourceTable"),
        c("_reason").alias("ErrorReason"),
        c("_error_col").alias("ErrorColumn"),
        F.lit("Silver").alias("ProcessingStage"),
        F.lit(RUN_ID).alias("BatchID"),
        F.lit(JOB).alias("PackageName"),
        F.current_timestamp().alias("ErrorDate"))
    rejected.coalesce(1).write.mode("overwrite").parquet(REJECTED + "/" + table)
    flagged.unpersist()

    silver_rows = spark.read.parquet(out).count()
    rate = round(100.0 * valid_n / total, 2) if total else None
    status = "OK" if silver_rows == valid_n else "MISMATCH"
    seconds = round(time.time() - t0, 1)
    print("%-16s total=%12d  valid=%12d  rejected=%8d  rate=%s  silver=%d  %s  %ss"
          % (table, total, valid_n, rejected_n, rate, silver_rows, status, seconds))
    return (table, RUN_ID, int(total), int(valid_n), int(rejected_n), rate, int(silver_rows), status, float(seconds))


results = [process(t) for t in TABLES]
summary = spark.createDataFrame(
    results,
    "table string, run_id string, total_records long, valid_records long, rejected_records long, "
    "success_rate double, silver_rows long, status string, seconds double")
summary.write.mode("overwrite").parquet(MONITORING)
summary.show(truncate=False)
spark.stop()

FactOnlineSales  total=    12627608  valid=    12527442  rejected=  100166  rate=99.21  silver=12527442  OK  224.5s
DimCustomer      total=       18869  valid=       18869  rejected=       0  rate=100.0  silver=18869  OK  4.4s
DimProduct       total=        2517  valid=        2517  rejected=       0  rate=100.0  silver=2517  OK  3.3s
DimStore         total=         306  valid=         306  rejected=       0  rate=100.0  silver=306  OK  2.8s
+---------------+--------------+-------------+-------------+----------------+------------+-----------+------+-------+
|table          |run_id        |total_records|valid_records|rejected_records|success_rate|silver_rows|status|seconds|
+---------------+--------------+-------------+-------------+----------------+------------+-----------+------+-------+
|FactOnlineSales|20261027115749|12627608     |12527442     |100166          |99.21       |12527442   |OK    |224.5  |
|DimCustomer    |20261027115749|18869        |18869        |0               |100.0